In [ ]:
import os
import requests
import time

from dotenv import load_dotenv

In [ ]:
# Loading the .env file from one folder above notebooks.
# Grab the value stored in STATS_API_KEY
# Reload the latest API key from .env
load_dotenv("../.env", override=True)

stats_api_key = os.getenv("STATS_API_KEY")

print("TheStatsAPI key loaded:", bool(stats_api_key))

# Instead of repeating the url, we store it as BASE_URL
BASE_URL = "https://api.thestatsapi.com/api/football"

# Our version of authentication.
headers = {
    "Authorization": f"Bearer {stats_api_key}"
}

In [ ]:
# We're packaging what we did yesterday into one reusable function
# Later we will be able to request and the function will handle everything and convert JSON response.
def get_api_data(url, max_retries = 5):
    wait_time = 5

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                headers = headers,
                timeout = 30
            )

        except requests.exceptions.RequestException as e:
            print(
                f"Connection error: {e}."
                f"Waiting {wait_time} seconds before retry "
                f"{attempt + 1}/{max_retries}..."
            )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            print(
                f"Rate limit reached. "
                f"Waiting {wait_time} seconds before retry "
                f"{attempt + 1}/{max_retries}"
            )

            time.sleep(wait_time)
            wait_time *= 2
            continue

        print("Request failed:", response.status_code)
        print(response.text[:500])
        return None

    print("Request failed after maximum retries.")
    return None


# This whole process is called exponential backoff. 
# We don't want to keep retrying because this will lead to incomplete data.
# We want each match to be saved in one single try, not multiple retries.

In [ ]:
# This function takes a match ID and builds the correct endpoint URL.
# The URL is then sent through the get_api_data function we created.
def get_match_details(match_id):
    url = f"{BASE_URL}/matches/{match_id}"
    return get_api_data(url)

In [ ]:
match_id = "mt_511976372"

# Takes a match ID and returns a clean match record.
match_details = get_match_details(match_id)

print(match_details)

In [ ]:
# Let's clean up our output so it's not just one messy Python dictionary.
match = match_details["data"]

print("Competition:", match["competition_name"])
print("Matchday:", match["matchday"])
print("Date:", match["utc_date"])
print("Home team:", match["home_team"]["name"])
print("Away team:", match["away_team"]["name"])
print("Score:", match["score"]["home"], "-", match["score"]["away"])
print("Venue:", match["venue"]["name"])
print("Referee:", match["referee"]["name"])
print("xG available:", match["xg_available"])
print("Odds available:", match["odds_available"])


In [ ]:
# Creating a function that retrieves statistics given a specific Match ID.
def get_match_statistics(match_id):
    url = f"{BASE_URL}/matches/{match_id}/stats"
    return get_api_data(url)

In [ ]:
match_statistics = get_match_statistics(match_id)
print(match_statistics.keys())

In [ ]:
# Cleaning up our match statistics
# Going from a messy Python dictionary to a clean format.

stats = match_statistics["data"]

print("Possession:", stats["overview"]["ball_possession"]["all"])
print("xG:", stats["overview"]["expected_goals"]["all"])
print("Shots:", stats["overview"]["total_shots"]["all"])
print("Shots on target:", stats["overview"]["shots_on_target"]["all"])
print("Passes:", stats["overview"]["passes"]["all"])
print("Corners:", stats["overview"]["corner_kicks"]["all"])

In [ ]:
# Creating a function to retrieve team lineups
def get_match_lineups(match_id):
    url = f"{BASE_URL}/matches/{match_id}/lineups"
    return get_api_data(url)

In [ ]:
# Testing lineups function
match_lineups = get_match_lineups(match_id)
print(match_lineups.keys())

In [ ]:
# Cleaning our lineup dictionary
lineups = match_lineups["data"]

print("Home team:", lineups["home"]["name"])
print("Home formation:", lineups["home"]["formation"])
print("Away team:", lineups["away"]["name"])
print("Away formation:", lineups["away"]["formation"])

In [ ]:
# Used to show not just the team formation, but the specific players in the lineup.
for player in lineups["home"]["starting_xi"]:
    print(
        player["jersey_number"],
        "-",
        player["name"],
        "-",
        player["position"]
    )

In [ ]:
# Creating a function to retrieve specific player's match stats.
def get_match_player_stats(match_id):
    url = f"{BASE_URL}/matches/{match_id}/player-stats"
    return get_api_data(url)  

In [ ]:
# Testing match player stats function.
match_player_stats = get_match_player_stats(match_id)
print(match_player_stats.keys())

In [ ]:
# Retrieving a team's players and looking at their match stats.
player_stats = match_player_stats["data"]

for player in player_stats:
    if player["team_id"] == "tm_9145" and player["started"]:
        print(
            player["player_name"],
            "- Rating:", player["rating"],
            "- Minutes:", player["minutes_played"]
        )

In [ ]:
# Creating a function to retrieve shotmaps
def get_match_shotmap(match_id):
    url = f"{BASE_URL}/matches/{match_id}/shotmap"
    return get_api_data(url)

In [ ]:
match_shotmap = get_match_shotmap(match_id)
print(match_shotmap.keys())

In [ ]:
# Cleaning up our shotmap dictionary.
shots = match_shotmap["data"]

for shot in shots[:5]:
    print(
        shot["minute"],
        "-",
        shot["player_name"],
        "-",
        shot["team_name"],
        "- x:", shot["x"],
        "- y:", shot["y"],
        "- Result:", shot["result"],
        "- xG:", shot["expected_goals"]
    )

In [ ]:
# Creating event timeline function.
def get_match_events(match_id):
    url = f"{BASE_URL}/matches/{match_id}/timeline"
    return get_api_data(url)

In [ ]:
# Testing our event timeline function.
match_events = get_match_events(match_id)
print(match_events.keys())

In [ ]:
# Testing to see how data is structured, we don't make to make any assumptions.
events_data = match_events["data"]
print(type(events_data))
print(events_data.keys())

In [ ]:
# Creating a shorter and more readable match timeline.
events = match_events["data"]["events"]

for event in events:
    if event["type"] in ["goal", "yellow_card", "red_card", "substitution"]:
        print(
            event["minute"],
            " -",
            event["type"],
            "-",
            event["team"]["name"] if event["team"] else None,
            "-",
            event["player"]["name"] if event["player"] else None
        )

In [ ]:
# Creating a function to retrieve match odds
def get_match_odds(match_id):
    url = f"{BASE_URL}/matches/{match_id}/odds"
    return get_api_data(url)

In [ ]:
# Testing our odds function
match_odds = get_match_odds(match_id)
print(match_odds.keys())

In [ ]:
# Now that we have seen that all our URL endpoints are working, we combine all of our helper functions into one single MASTER function that can retrieve all our info with one request.

def get_full_match_data(match_id):
    return {
        "details": get_match_details(match_id),
        "statistics": get_match_statistics(match_id),
        "lineups": get_match_lineups(match_id),
        "player_stats": get_match_player_stats(match_id),
        "shotmap": get_match_shotmap(match_id),
        "events": get_match_events(match_id),
        "odds": get_match_odds(match_id)
    }

In [ ]:
# Testing out our MASTER function
full_match_data = get_full_match_data(match_id)
print(full_match_data.keys())

In [ ]:
get_full_match_data("mt_511976372")

In [ ]:
# Automating pulling match IDs
# No more hard coding specific matches.

def get_season_matches(competition_id, season_id):
    all_matches = []
    page = 1

    while True:
        url = (
            f"{BASE_URL}/matches"
            f"?competition_id={competition_id}"
            f"&season_id={season_id}"
            f"&page={page}"
        )

        response = get_api_data(url)

        if response is None:
            print(f"Failed to retrieve page {page}")
            return None

        matches = response["data"]
        all_matches.extend(matches)

        total_pages = response["meta"]["total_pages"]

        print(
            f"Page {page}/{total_pages} collected "
            f"({len(matches)} matches)"
        )

        if page >= total_pages:
            break

        page += 1
        time.sleep(5)

    return {
        "data": all_matches,
        "meta": {
            "total": len(all_matches),
            "total_pages": total_pages
        }
    }

In [ ]:
# Testing getting season matches function.
season_matches = get_season_matches(
    "comp_3039",
    "sn_8406098"
)

print(season_matches.keys())

In [ ]:
# Looking to see how many matches were returned from the URL
print("Matches returned:", len(season_matches["data"]))
print("Meta:", season_matches["meta"])

In [ ]:
for match in season_matches["data"][:10]:
    print(
        match["id"],
        "-",
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["status"]
    )

In [ ]:
# Building pagination, each page fits 20 matches, we currently have a total of 50, so we need 3 pages
def get_all_season_matches(competition_id, season_id):
    all_matches = []

    # We start at page 1 and adds the matches into our array: all_matches
    page = 1

    while True:
        url = (
            f"{BASE_URL}/matches"
            f"?competition_id={competition_id}"
            f"&season_id={season_id}"
            f"&page={page}"
        )

        response = get_api_data(url)

        if response is None:
            break

        # extend() function takes all the matches from that page and adds them to our master list
        all_matches.extend(response["data"])

        # API told us there are 3 pages, so Python will separate into 20 matches in the first 2 pages, then 10 in the last page and then combine them into one list of 50 matches.
        if page >= response["meta"]["total_pages"]:
            break

        page += 1

        # Gives the API a pause so we don't hit a rate limit.
        time.sleep(1)

    return all_matches

In [ ]:
# Testing our match getting function
all_season_matches = get_all_season_matches(
    "comp_3039",
    "sn_8406098"
)

print("Total matches collected:", len(all_season_matches))

In [ ]:
# Our function is showing us finished AND future scheduled matches
# We want to separate matches that have already finished and the ones that haven't been played yet.

finished_matches = [
    match for match in all_season_matches
    if match["status"] == "finished"
]

scheduled_matches = [
    match for match in all_season_matches
    if match["status"] == "scheduled"
]

print("Finished matches:", len(finished_matches))
print("Scheduled matches:", len(scheduled_matches))

In [ ]:
# Let's take a look at the finished matches.

for match in finished_matches:
    print(
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["score"]["home"],
        "-",
        match["score"]["away"]
    )

In [ ]:
# Create a lighter function to inspect past matches (historical modeling)

def get_model_match_data(match_id):
    full_data = {}

    full_data["details"] = get_match_details(match_id)
    time.sleep(2)

    full_data["statistics"] = get_match_statistics(match_id)
    time.sleep(2)

    full_data["lineups"] = get_match_lineups(match_id)
    time.sleep(2)

    full_data["player_stats"] = get_match_player_stats(match_id)
    time.sleep(2)

    full_data["odds"] = get_match_odds(match_id)

    missing = [
        section
        for section, value in full_data.items()
        if value is None
    ]

    if missing:
        print("Incomplete match data. Missing:", missing)
        return None

    return full_data

In [ ]:
# Testing our function on one completed match.
test_model_data = get_model_match_data(finished_matches[0]["id"])
print(test_model_data.keys())

In [ ]:
# Saving our matches

import json
from pathlib import Path

In [ ]:
RAW_MATCH_DIR = Path("../data/raw/matches")
RAW_MATCH_DIR.mkdir(parents = True, exist_ok = True)
print(RAW_MATCH_DIR)

In [ ]:
# Creating a function that saves the match into the correct folder.

def save_match_data(match_id, match_data, save_dir):
    # Creating the filename.
    file_path = save_dir / f"{match_id}.json"

    # We open the file and write (w)
    # utf-8 makes sure special characters are also stored.
    with open(file_path, "w", encoding = "utf-8") as f:
        # match_data is a Python dictionary, so json.dump takes it and writes it into a .json file.
        json.dump(match_data, f, indent = 2, ensure_ascii = False)

    return file_path

In [ ]:
# Save our first match

test_match_id = finished_matches[0]["id"]

saved_path = save_match_data(
    test_match_id,
    test_model_data,
    RAW_MATCH_DIR / "2026_27"
)

print("Saved to:", saved_path)

In [ ]:
# Automating the entire saving matches process.

# Going through every finished match in our list.
for match in finished_matches:
    match_id = match["id"]
    file_path = RAW_MATCH_DIR / "2026_27" / f"{match_id}.json"

    # If the match has already been saved, we skip that match.
    if file_path.exists():
        print("Skipping:", match_id, "- already saved")
        continue

    # If the match doesn't already exist, we save the match.
    print(
        "Collecting:",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"]
    )

    # Pulls the model data
    match_data = get_model_match_data(match_id)

    # Saves the data
    if match_data is not None:
        save_match_data(
            match_id, 
            match_data,
            RAW_MATCH_DIR / "2026_27"
        )
        
        print("Saved:", match_id)

    # 2 second pause to avoid hitting the rate limit.
    time.sleep(2)

In [ ]:
# Let's check if everything was saved properly

for file_path in RAW_MATCH_DIR.glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section, value in data.items()
        if value is None
    ]

    if missing:
        print(file_path.name, "- MISSING:", missing)
    else:
        print(file_path.name, "- COMPLETE")

In [ ]:
import pandas as pd

In [ ]:
# A JSON file might be too nested for machine learning purposes.
# This function takes a JSON dictionary and turns it into something flat where each match becomes a row in our future dataset.
def extract_match_row(match_data):
    details = match_data["details"]["data"]
    stats = match_data["statistics"]["data"]
    lineups = match_data["lineups"]["data"]

    return {
        "match_id": details["id"],
        "date": details["utc_date"],
        "matchday": details["matchday"],

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": stats["overview"]["expected_goals"]["all"]["home"],
        "away_xg": stats["overview"]["expected_goals"]["all"]["away"],

        "home_shots": stats["overview"]["total_shots"]["all"]["home"],
        "away_shots": stats["overview"]["total_shots"]["all"]["away"],

        "home_shots_on_target": stats["overview"]["shots_on_target"]["all"]["home"],
        "away_shots_on_target": stats["overview"]["shots_on_target"]["all"]["away"],

        "home_possession": stats["overview"]["ball_possession"]["all"]["home"],
        "away_possession": stats["overview"]["ball_possession"]["all"]["away"],

        "home_passes": stats["overview"]["passes"]["all"]["home"],
        "away_passes": stats["overview"]["passes"]["all"]["away"],

        "home_corners": stats["overview"]["corner_kicks"]["all"]["home"],
        "away_corners": stats["overview"]["corner_kicks"]["all"]["away"],

        "home_formation": lineups["home"]["formation"],
        "away_formation": lineups["away"]["formation"],

        "winner": details["score"]["winner"]
    }

In [ ]:
# Now each row will correspond to a JSON file

match_rows = []

for file_path in (RAW_MATCH_DIR / "2026_27").glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        match_data = json.load(f)

    row = extract_match_row(match_data)
    match_rows.append(row)


matches_df = pd.DataFrame(match_rows)

In [ ]:
# Inspecting our match-level DataFrame
print(matches_df.shape)
matches_df.head()

In [ ]:
# Check to see if all our variables have the correct data type.
matches_df.info()

In [ ]:
# We see if there are any null valuables in our columns.
matches_df.isnull().sum()

In [ ]:
matches_df["date"] = pd.to_datetime(matches_df["date"], utc = True)
matches_df.info()

In [ ]:
# Moving from raw data to processed data.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

In [ ]:
processed_file = PROCESSED_DIR / "matches.csv"

matches_df.to_csv(
    processed_file,

    # Prevents pandas from adding unnecessary columns containing 0,1,2,3,..
    index = False
)

print("Saved to:", processed_file)

In [ ]:
# Moving onto historical data (Past seasons)
PREMIER_LEAGUE_ID = "comp_3039"

SEASONS = {
    "2026_27": "sn_8406098",
    "2025_26": "sn_6125938",
    "2024_25": "sn_3057848",
    "2023_24": "sn_606923",
    "2022_23": "sn_654318",
    "2021_22": "sn_699021",
    "2020_21": "sn_477656"
}

In [ ]:
collect_season_baseline_data(
    "2022_23",
    finished_2022_23
)

In [ ]:
matches_2022_23 = season_2022_23_matches["data"]

finished_2022_23 = [
    match for match in matches_2022_23
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_2022_23))

test_match = finished_2022_23[0]

print("Test match ID:", test_match["id"])
print(
    test_match["home_team"]["name"],
    "vs",
    test_match["away_team"]["name"]
)

In [ ]:
# Testing on a past season to see if all matches are returned to us from the API.
# For a completed season, the total amount of games is 380.
matches_2025_26 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2025_26"]
)

print("Matches found:", len(matches_2025_26))

In [ ]:
from collections import Counter

status_counts = Counter(
    match["status"] for match in matches_2025_26
)

print(status_counts)

In [ ]:
# Used as a sanity check to make sure we have pulled matches from the correct season (2025-26)
for match in matches_2025_26[:5]:
    print(
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["status"]
    )

In [ ]:
# Creating a historical collector.
def get_baseline_match_data(match_id):
    match_data = {}

    match_data["details"] = get_match_details(match_id)
    time.sleep(2)

    match_data["statistics"] = get_match_statistics(match_id)
    time.sleep(2)

    match_data["odds"] = get_match_odds(match_id)

    # Details are required because we need the actual match/result.
    if match_data["details"] is None:
        print("Missing match details. Skipping:", match_id)
        return None

    # Statistics and odds can be missing and will be handled later as NaN.
    missing = [
        section
        for section in ["statistics", "odds"]
        if match_data[section] is None
    ]

    if missing:
        print("Saving with missing sections:", missing)

    return match_data

In [ ]:
# Creating folders for each season.
SEASON_DIRS = {}

# Looping through the seasons we listed earlier.
# Then creating a path for each season
for season_name in SEASONS:
    season_dir = RAW_MATCH_DIR / season_name
    season_dir.mkdir(parents = True, exist_ok = True)
    SEASON_DIRS[season_name] = season_dir

print(SEASON_DIRS)

In [ ]:
# Building a collector for the historical seasons.

def collect_season_baseline_data(season_name, matches):
    season_dir = SEASON_DIRS[season_name]

    total_matches = len(matches)

    # Adds a progress counter, just to keep track of how many matches are left to save.
    for i, match in enumerate(matches, start = 1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        # The collrector is resumable, so if it stops collecting today, you can continue collecting tomorrow.
        if file_path.exists():
            print(f"[{i}/{total_matches}] Skipping {match_id} - already saved")
            continue

        print(
            f"[{i}/{total_matches}] Collecting:",
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"]
        )

        match_data = get_baseline_match_data(match_id)

        if match_data is not None:
            with open(file_path, "w", encoding = "utf-8") as f:
                json.dump(match_data, f, indent = 2, ensure_ascii = False)

            print("Saved:", match_id)
        else:
            print("Not saved because data was incomplete:", match_id)

        time.sleep(5)

In [ ]:
# Testing to see if the historical seasons collector works.
collect_season_baseline_data(
    "2025_26",
    matches_2025_26[:3]
)

In [ ]:
# Retrieve the latest 2026/27 Premier League matches
season_2026_27_matches = get_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2026_27"]
)

# Check whether the API returned data
if season_2026_27_matches is not None:
    print("API response received!")
    print("Total matches:", len(season_2026_27_matches["data"]))
else:
    print("API request failed.")

In [ ]:
from collections import Counter

matches_2026_27 = season_2026_27_matches["data"]

status_counts = Counter(
    match["status"] for match in matches_2026_27
)

print("2026/27 Premier League Match Statuses:")
for status, count in status_counts.items():
    print(f"{status}: {count}")

print("\nTotal matches:", len(matches_2026_27))

In [ ]:
# Filter the 2026/27 matches to completed games only
finished_2026_27 = [
    match
    for match in season_2026_27_matches["data"]
    if match["status"] == "finished"
]

print("Completed matches:", len(finished_2026_27))

In [ ]:
from pathlib import Path

season_dir = SEASON_DIRS["2026_27"]

# Identify matches already saved locally
saved_match_ids = {
    file.stem
    for file in season_dir.glob("*.json")
}

# Find completed matches not yet downloaded
missing_matches = [
    match
    for match in finished_2026_27
    if str(match["id"]) not in saved_match_ids
]

print("Completed matches:", len(finished_2026_27))
print("Existing JSON files:", len(saved_match_ids))
print("Missing completed matches:", len(missing_matches))

print("\nFirst 5 missing matches:")
for match in missing_matches[:5]:
    print(
        match["id"],
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"]
    )

In [ ]:
collect_season_baseline_data(
    "2026_27",
    finished_2026_27
)

In [ ]:
PLAYER_STATS_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "player_stats"
    / "premier_league"
    / "2026_27"
)

PLAYER_STATS_DIR.mkdir(parents=True, exist_ok=True)


def collect_player_stats(matches):
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]

        file_path = PLAYER_STATS_DIR / f"{match_id}.json"
        temp_path = PLAYER_STATS_DIR / f"{match_id}.json.tmp"

        # Skip existing files only if they contain valid JSON
        if file_path.exists():
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    existing_data = json.load(f)

                if (
                    isinstance(existing_data, dict)
                    and isinstance(existing_data.get("data"), list)
                    and len(existing_data["data"]) > 0
                ):
                    print(f"[{i}/{total_matches}] Skipping {match_id}")
                    continue

            except (json.JSONDecodeError, UnicodeDecodeError):
                print(f"[{i}/{total_matches}] Invalid file - recollecting")

        print(f"[{i}/{total_matches}] Collecting player stats: {match_id}")

        url = f"{BASE_URL}/matches/{match_id}/player-stats"
        response = get_api_data(url)

        if (
            response is not None
            and isinstance(response.get("data"), list)
            and len(response["data"]) > 0
        ):
            try:
                with open(temp_path, "w", encoding="utf-8") as f:
                    json.dump(response, f, indent=2, ensure_ascii=False)

                temp_path.replace(file_path)

                print(f"Saved {len(response['data'])} player records")

            finally:
                if temp_path.exists():
                    temp_path.unlink()
        else:
            print("No valid player statistics returned:", match_id)

        time.sleep(5)

In [ ]:
collect_player_stats(finished_2026_27)

In [ ]:
valid_matches = 0
total_player_records = 0
missing_matches = []

for match in finished_2026_27:
    match_id = match["id"]
    file_path = PLAYER_STATS_DIR / f"{match_id}.json"

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data.get("data"), list) and data["data"]:
            valid_matches += 1
            total_player_records += len(data["data"])
        else:
            missing_matches.append(match_id)

    except (FileNotFoundError, json.JSONDecodeError):
        missing_matches.append(match_id)

print("Completed matches:", len(finished_2026_27))
print("Matches with valid player statistics:", valid_matches)
print("Total player-match records:", total_player_records)
print("Missing or invalid matches:", len(missing_matches))

if missing_matches:
    print("Matches needing attention:", missing_matches)

In [ ]:
# Inspect all available player-statistics fields
all_fields = {}

for file_path in PLAYER_STATS_DIR.glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        match_data = json.load(f)

    for player in match_data.get("data", []):
        for category, values in player.items():

            if isinstance(values, dict):
                if category not in all_fields:
                    all_fields[category] = set()

                all_fields[category].update(values.keys())

            else:
                if "player_info" not in all_fields:
                    all_fields["player_info"] = set()

                all_fields["player_info"].add(category)

for category, fields in all_fields.items():
    print(f"\n--- {category.upper()} ---")

    for field in sorted(fields):
        print(field)

In [ ]:
import pandas as pd
import json

player_records = []

for file_path in sorted(PLAYER_STATS_DIR.glob("*.json")):

    match_id = file_path.stem

    with open(file_path, "r", encoding="utf-8") as f:
        match_data = json.load(f)

    for player in match_data.get("data", []):

        # Flatten nested player statistics into columns
        record = pd.json_normalize(player, sep="_").to_dict(
            orient="records"
        )[0]

        record["match_id"] = match_id

        player_records.append(record)

# Combine all player-match records
player_df = pd.DataFrame(player_records)

# Check for duplicate player records within a match
duplicate_count = player_df.duplicated(
    subset=["match_id", "player_id"]
).sum()

print("Total player-match records:", len(player_df))
print("Unique players:", player_df["player_id"].nunique())
print("Total columns:", len(player_df.columns))
print("Duplicate player-match records:", duplicate_count)

print("\nColumns:")
print(player_df.columns.tolist())

In [ ]:
import numpy as np

# Ensure numeric columns have numeric data types
numeric_columns = [
    "minutes_played",
    "passing_total_passes",
    "passing_accurate_passes",
    "shooting_goals",
    "shooting_total_shots",
    "shooting_shots_on_target",
    "shooting_expected_goals",
    "shooting_expected_assists",
]

for col in numeric_columns:
    player_df[col] = pd.to_numeric(
        player_df[col], errors="coerce"
    )

# Calculate passing accuracy (%)
player_df["passing_accuracy"] = np.where(
    player_df["passing_total_passes"] > 0,
    (
        player_df["passing_accurate_passes"]
        / player_df["passing_total_passes"]
    ) * 100,
    np.nan
)

# Save the processed dataset
output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "premier_league_player_stats_2026_27.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

player_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(player_df))
print("Columns:", len(player_df.columns))
print("Passing accuracy range:",
      player_df["passing_accuracy"].min(),
      "to",
      player_df["passing_accuracy"].max())

In [ ]:
# Testing the collector on 3 matches worked, so we can get rid of the [:3] limit.
collect_season_baseline_data(
    "2025_26",
    matches_2025_26
)

In [ ]:
# Verify the 2025/26 season folder has 380 JSON files.
season_2025_dir = SEASON_DIRS["2025_26"]
saved_files = list(season_2025_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

In [ ]:
# Perform an integrity check
# Making sure every JSON file contains the sections we want to look at

incomplete_files = []

for file_path in SEASON_DIRS["2025_26"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", 380)
print("Incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2025/26 files are complete.")

In [ ]:
# Now collecting historical data from the 2024/25 season
matches_2024_25 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2024_25"]
)

print("Matches found:", len(matches_2024_25))

In [ ]:
# Our last cell returned 381 matches, so we have to check for any duplicates

status_counts_2024_25 = Counter(
    match["status"] for match in matches_2024_25
)

print(status_counts_2024_25)

In [ ]:
match_ids_2024_25 = [
    match["id"]
    for match in matches_2024_25
]

print("Total matches:", len(match_ids_2024_25))
print("Unique match IDs:", len(set(match_ids_2024_25)))

In [ ]:
for match in matches_2024_25:
    if match["status"] != "finished":
        print(
            match["id"],
            match["matchday"],
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"],
            "-",
            match["status"]
        )

In [ ]:
# We found the duplicate record, let's see if the match ID shows that it was finished
for match in matches_2024_25:
    if (
        match["home_team"]["name"] == "Manchester City"
        and match["away_team"]["name"] == "Bournemouth"
    ):
        print(
            match["id"],
            "- Matchday:", match["matchday"],
            "- Status:", match["status"]
        )

In [ ]:
# Creating a clean list
# Removing the postoned match ID

matches_2024_25_clean = [
    match
    for match in matches_2024_25
    if match["status"] == "finished"
]

print("Original matches:", len(matches_2024_25))
print("Finished matches:", len(matches_2024_25_clean))

In [ ]:
# Run the collector function for the 2024/25 season.

collect_season_baseline_data(
    "2024_25",
    matches_2024_25_clean
)

In [ ]:
# Verify that all matches in season 2024/25 saved properly.

season_2024_dir = SEASON_DIRS["2024_25"]

saved_files = list(season_2024_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

In [ ]:
# Check for incomplete files
incomplete_files = []

for file_path in SEASON_DIRS["2024_25"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", len(saved_files))
print("Incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2024/25 files are complete.")

In [ ]:
# Start collecting historical data from season 2023/24
# First conduct an initial check on that season.
matches_2023_24 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2023_24"]
)

print("Matches found:", len(matches_2023_24))

In [ ]:
status_counts_2023_24 = Counter(
    match["status"] for match in matches_2023_24
)

print(status_counts_2023_24)

In [ ]:
# This season doesn't need any filtering
# Go ahead and collect all the data from that season.

collect_season_baseline_data(
    "2023_24",
    matches_2023_24
)

In [ ]:
# ---------------------------------------------------------
# GET 2026/27 PREMIER LEAGUE MATCH LIST
# ---------------------------------------------------------

matches_2026_27 = get_all_season_matches(
    "comp_3039",
    "sn_8406098"
)

print(
    "2026/27 Premier League matches found:",
    len(matches_2026_27)
)

In [ ]:
print("Total returned:", len(matches_2026_27))
print()

for match in matches_2026_27[:10]:
    print(
        match.get("id"),
        match.get("date"),
        match.get("status"),
        match.get("home_team", {}).get("name"),
        "vs",
        match.get("away_team", {}).get("name")
    )

In [ ]:
from collections import Counter

status_counts = Counter(
    match.get("status")
    for match in matches_2026_27
)

print("STATUS COUNTS")
print()

for status, count in status_counts.items():
    print(status, count)

In [ ]:
# ---------------------------------------------------------
# KEEP ONLY FINISHED 2026/27 PREMIER LEAGUE MATCHES
# ---------------------------------------------------------

finished_matches_2026_27 = [
    match
    for match in matches_2026_27
    if match.get("status") == "finished"
]


print(
    "Finished 2026/27 Premier League matches:",
    len(finished_matches_2026_27)
)

In [ ]:
# ---------------------------------------------------------
# COLLECT / RESUME FINISHED 2026/27 MATCHES
# ---------------------------------------------------------

collect_season_baseline_data(
    "2026_27",
    finished_matches_2026_27
)

In [ ]:
import json
import pandas as pd
from pathlib import Path


# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------

RAW_MATCH_DIR = Path("../data/raw/matches")

PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# SAFE STAT EXTRACTOR
# ---------------------------------------------------------

def get_stat(statistics, team_side, stat_name):

    if statistics is None:
        return None

    if isinstance(statistics, dict):
        statistics = statistics.get(
            "data",
            statistics
        )

    if not isinstance(statistics, dict):
        return None

    team_stats = statistics.get(
        team_side,
        {}
    )

    if not isinstance(team_stats, dict):
        return None

    value = team_stats.get(
        stat_name
    )

    return value


# ---------------------------------------------------------
# PROCESS ONE RAW MATCH FILE
# ---------------------------------------------------------

def process_match_file(file_path):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # Only completed matches
    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "statistics"
    )


    # -----------------------------------------------------
    # TEAM INFORMATION
    # -----------------------------------------------------

    home_team_data = details.get(
        "home_team",
        {}
    )

    away_team_data = details.get(
        "away_team",
        {}
    )


    home_team = (
        home_team_data.get("name")
        if isinstance(
            home_team_data,
            dict
        )
        else None
    )

    away_team = (
        away_team_data.get("name")
        if isinstance(
            away_team_data,
            dict
        )
        else None
    )


    # -----------------------------------------------------
    # SCORE
    # -----------------------------------------------------

    home_goals = details.get(
        "home_score"
    )

    away_goals = details.get(
        "away_score"
    )


    # Some API responses may store score differently
    if isinstance(home_goals, dict):

        home_goals = (
            home_goals.get("current")
            or home_goals.get("display")
        )


    if isinstance(away_goals, dict):

        away_goals = (
            away_goals.get("current")
            or away_goals.get("display")
        )


    # -----------------------------------------------------
    # WINNER
    # -----------------------------------------------------

    winner = None

    if (
        home_goals is not None
        and away_goals is not None
    ):

        try:

            if home_goals > away_goals:
                winner = "H"

            elif away_goals > home_goals:
                winner = "A"

            else:
                winner = "D"

        except TypeError:
            winner = None


    # -----------------------------------------------------
    # FOLDER INFORMATION
    #
    # .../matches/competition/season/file.json
    # -----------------------------------------------------

    season = file_path.parent.name

    competition = (
        file_path
        .parent
        .parent
        .name
    )


    # -----------------------------------------------------
    # BUILD ROW
    # -----------------------------------------------------

    row = {

        "match_id":
            details.get("id"),

        "date":
            details.get("date"),

        "season":
            season,

        "competition":
            competition,

        "matchday":
            details.get("matchday"),

        "home_team":
            home_team,

        "away_team":
            away_team,

        "home_goals":
            home_goals,

        "away_goals":
            away_goals,

        "home_xg":
            get_stat(
                statistics,
                "home",
                "expected_goals"
            ),

        "away_xg":
            get_stat(
                statistics,
                "away",
                "expected_goals"
            ),

        "home_shots":
            get_stat(
                statistics,
                "home",
                "shots"
            ),

        "away_shots":
            get_stat(
                statistics,
                "away",
                "shots"
            ),

        "home_shots_on_target":
            get_stat(
                statistics,
                "home",
                "shots_on_target"
            ),

        "away_shots_on_target":
            get_stat(
                statistics,
                "away",
                "shots_on_target"
            ),

        "home_possession":
            get_stat(
                statistics,
                "home",
                "ball_possession"
            ),

        "away_possession":
            get_stat(
                statistics,
                "away",
                "ball_possession"
            ),

        "home_passes":
            get_stat(
                statistics,
                "home",
                "passes"
            ),

        "away_passes":
            get_stat(
                statistics,
                "away",
                "passes"
            ),

        "home_corners":
            get_stat(
                statistics,
                "home",
                "corners"
            ),

        "away_corners":
            get_stat(
                statistics,
                "away",
                "corners"
            ),

        "winner":
            winner
    }


    return row


# ---------------------------------------------------------
# PROCESS ALL RAW JSON FILES
# ---------------------------------------------------------

rows = []


for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATE MATCH IDS
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# SORT
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# CHECK RESULTS
# ---------------------------------------------------------

print(
    "Historical shape:",
    historical_df.shape
)

print()

print(
    "COMPETITION COUNTS"
)

print()

print(
    historical_df[
        "competition"
    ]
    .value_counts()
)

print()

print(
    "Premier League by season"
)

print()

print(
    historical_df[
        historical_df[
            "competition"
        ]
        == "premier_league"
    ][
        "season"
    ]
    .value_counts()
    .sort_index()
)

print()

print(
    "Saved:",
    output_path
)

In [ ]:
from pathlib import Path

RAW_MATCH_DIR = Path("../data/raw/matches")

In [ ]:
# =========================================================
# RAW DATA COVERAGE AUDIT
# =========================================================

from collections import defaultdict
import json
import pandas as pd

audit = []

for file_path in RAW_MATCH_DIR.rglob("*.json"):

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except:
        continue

    details = data.get("details")
    stats = data.get("statistics")

    if isinstance(details, dict):
        details = details.get("data", details)

    if isinstance(stats, dict):
        stats_data = stats.get("data", stats)
    else:
        stats_data = None

    competition = file_path.parent.parent.name
    season = file_path.parent.name

    has_stats = (
        isinstance(stats_data, dict)
        and len(stats_data) > 0
    )

    home_stats = (
        stats_data.get("home", {})
        if has_stats else {}
    )

    audit.append({
        "competition": competition,
        "season": season,
        "file": file_path.name,
        "has_statistics": has_stats,
        "has_xg": (
            isinstance(home_stats, dict)
            and home_stats.get("expected_goals") is not None
        ),
        "has_shots": (
            isinstance(home_stats, dict)
            and home_stats.get("shots") is not None
        ),
        "has_possession": (
            isinstance(home_stats, dict)
            and home_stats.get("ball_possession") is not None
        ),
        "has_passes": (
            isinstance(home_stats, dict)
            and home_stats.get("passes") is not None
        ),
        "has_corners": (
            isinstance(home_stats, dict)
            and home_stats.get("corners") is not None
        )
    })

audit_df = pd.DataFrame(audit)

# Premier League only
pl_audit = audit_df[
    audit_df["competition"] == "premier_league"
].copy()

summary = (
    pl_audit
    .groupby("season")
    .agg(
        raw_files=("file", "count"),
        files_with_stats=("has_statistics", "sum"),
        files_with_xg=("has_xg", "sum"),
        files_with_shots=("has_shots", "sum"),
        files_with_possession=("has_possession", "sum"),
        files_with_passes=("has_passes", "sum"),
        files_with_corners=("has_corners", "sum"),
    )
)

print(summary.to_string())

In [ ]:
from pathlib import Path
import json
import pprint

RAW_MATCH_DIR = Path("../data/raw/matches")

# Grab one 2023/24 PL match
files = list(
    (RAW_MATCH_DIR / "premier_league" / "2023_24").glob("*.json")
)

print("Files found:", len(files))

sample_file = files[0]

print("\nSample file:")
print(sample_file)

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("\nTOP-LEVEL KEYS:")
print(sample.keys())

print("\nSTATISTICS TYPE:")
print(type(sample.get("statistics")))

print("\nSTATISTICS CONTENT:")
pprint.pp(sample.get("statistics"), depth=6)

In [ ]:
from pathlib import Path
import json
import pandas as pd

RAW_MATCH_DIR = Path("../data/raw/matches")

rows = []

for file_path in (
    RAW_MATCH_DIR / "premier_league"
).rglob("*.json"):

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        continue

    stats = data.get("statistics", {})

    if isinstance(stats, dict):
        stats = stats.get("data", stats)

    overview = (
        stats.get("overview", {})
        if isinstance(stats, dict)
        else {}
    )

    def overview_stat(name):
        try:
            return overview[name]["all"]["home"]
        except (KeyError, TypeError):
            return None

    rows.append({
        "season": file_path.parent.name,
        "xg": overview_stat("expected_goals"),
        "possession": overview_stat("ball_possession"),
        "shots": overview_stat("shots"),
        "shots_on_target": overview_stat("shots_on_target"),
        "passes": overview_stat("passes"),
        "corners": overview_stat("corners"),
    })

audit = pd.DataFrame(rows)

summary = (
    audit.groupby("season")
    .agg(
        matches=("season", "size"),
        xg=("xg", "count"),
        possession=("possession", "count"),
        shots=("shots", "count"),
        shots_on_target=("shots_on_target", "count"),
        passes=("passes", "count"),
        corners=("corners", "count"),
    )
)

print(summary.to_string())

In [ ]:
stats = sample["statistics"]["data"]

print("STATISTICS SECTIONS:")
print(stats.keys())

print("\nOVERVIEW STAT NAMES:")
print(stats["overview"].keys())

In [ ]:
from pathlib import Path
import json
import pandas as pd

RAW_MATCH_DIR = Path("../data/raw/matches")

rows = []

for file_path in (RAW_MATCH_DIR / "premier_league").rglob("*.json"):

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        continue

    stats = data.get("statistics", {})

    if isinstance(stats, dict):
        stats = stats.get("data", stats)

    overview = (
        stats.get("overview", {})
        if isinstance(stats, dict)
        else {}
    )

    def get_overview_stat(name):
        try:
            value = overview[name]["all"]["home"]
            return value
        except (KeyError, TypeError):
            return None

    rows.append({
        "season": file_path.parent.name,
        "xg": get_overview_stat("expected_goals"),
        "possession": get_overview_stat("ball_possession"),
        "shots": get_overview_stat("total_shots"),
        "shots_on_target": get_overview_stat("shots_on_target"),
        "passes": get_overview_stat("passes"),
        "corners": get_overview_stat("corner_kicks"),
    })

audit = pd.DataFrame(rows)

summary = (
    audit.groupby("season")
    .agg(
        matches=("season", "size"),
        xg=("xg", "count"),
        possession=("possession", "count"),
        shots=("shots", "count"),
        shots_on_target=("shots_on_target", "count"),
        passes=("passes", "count"),
        corners=("corners", "count"),
    )
)

print(summary.to_string())

In [ ]:
# Verify all matches were saved.

season_2023_dir = SEASON_DIRS["2023_24"]

saved_files = list(season_2023_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

In [ ]:
incomplete_files = []

for file_path in SEASON_DIRS["2023_24"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", len(saved_files))
print("incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2023/24 files are complete.")

In [ ]:
import shutil

current_season_dir = SEASON_DIRS["2026_27"]

for file_path in RAW_MATCH_DIR.glob("*.json"):
    destination = current_season_dir / file_path.name

    shutil.move(str(file_path), str(destination))

    print("Moved:", file_path.name)

In [ ]:
def get_stat(overview, stat_name, side):
    stat = overview.get(stat_name)

    if stat is None:
        return None

    full_match = stat.get("all")

    if full_match is None:
        return None

    return full_match.get(side)

In [ ]:
missing_xg_matches = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        details = match_data["details"]["data"]
        overview = match_data["statistics"]["data"]["overview"]

        expected_goals = overview.get("expected_goals")

        if expected_goals is None or expected_goals.get("all") is None:
            missing_xg_matches.append({
                "season": season_name,
                "match_id": details["id"],
                "home_team": details["home_team"]["name"],
                "away_team": details["away_team"]["name"]
            })

print("Matches missing xG:", len(missing_xg_matches))

for match in missing_xg_matches:
    print(match)

In [ ]:
# Helper function for matches with no xG.
def get_stat(stats, stat_name, side):
    stat = stats["overview"].get(stat_name)

    if stat is None or stat.get("all") is None:
        return None

    return stat["all"][side]

In [ ]:
def extract_historical_match_row(match_data):
    details = match_data["details"]["data"]
    stats = match_data["statistics"]["data"]

    return {
        "match_id": details["id"],
        "date": details["utc_date"],
        "matchday": details["matchday"],

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": get_stat(stats, "expected_goals", "home"),
        "away_xg": get_stat(stats, "expected_goals", "away"),

        "home_shots": get_stat(stats, "total_shots", "home"),
        "away_shots": get_stat(stats, "total_shots", "away"),

        "home_shots_on_target": get_stat(stats, "shots_on_target", "home"),
        "away_shots_on_target": get_stat(stats, "shots_on_target", "away"),

        "home_possession": get_stat(stats, "ball_possession", "home"),
        "away_possession": get_stat(stats, "ball_possession", "away"),

        "home_passes": get_stat(stats, "passes", "home"),
        "away_passes": get_stat(stats, "passes", "away"),

        "home_corners": get_stat(stats, "corner_kicks", "home"),
        "away_corners": get_stat(stats, "corner_kicks", "away"),

        "winner": details["score"]["winner"]
    }

In [ ]:
historical_rows = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        row = extract_historical_match_row(match_data)
        row["season"] = season_name

        historical_rows.append(row)

print("Historical rows collected:", len(historical_rows))

In [ ]:
# Turning our historical seasons rows into a DataFrame

historical_df = pd.DataFrame(historical_rows)

historical_df.shape

In [ ]:
historical_df.head()

In [ ]:
# Checking to see if there are any null values to clean up.
historical_df.isnull().sum()

In [ ]:
historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc = True
)

historical_df.dtypes

In [ ]:
# Conduct a final quality check.

print("Total matches:", len(historical_df))
print("Duplicate match IDs:", historical_df["match_id"].duplicated().sum())

print("\nMatches per season:")
print(historical_df["season"].value_counts())

print("\nDate range:")
print(historical_df["date"].min())
print(historical_df["date"].max())

In [ ]:
# Saving the processed historical dataset.

historical_file = PROCESSED_DIR / "historical_matches.csv"

historical_df.to_csv(
    historical_file,
    index = False
)

print("Saved to:", historical_file)

In [ ]:
# Inspect a historical match's odds structure
# Will help us decide what fields to extract
# Shows us how the API organizes bookmakers, markets, and prices.
sample_file = next(SEASON_DIRS["2023_24"].glob("*.json"))

with open(sample_file, "r", encoding = "utf-8") as f:
    sample_match = json.load(f)

print(json.dumps(sample_match["odds"], indent = 2)[:5000])

In [ ]:
# Check if all 1,140 matches have Bet365 odds
# Showing which bookmakers are available across the dataset and how often they appear.
bookmaker_counts = {}

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding = "utf-8") as f:
            match_data = json.load(f)

        bookmakers = match_data["odds"]["data"]["bookmakers"]

        for bookmaker in bookmakers:
            name = bookmaker["bookmaker"]

            bookmaker_counts[name] = bookmaker_counts.get(name, 0) + 1

print(bookmaker_counts)


In [ ]:
# Make sure match odds are present for all 1,140 matches.

missing_match_odds = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding = "utf-8") as f:
            match_data = json.load(f)

        details = match_data["details"]["data"]
        bookmakers = match_data["odds"]["data"]["bookmakers"]

        bet365 = bookmakers[0]
        match_odds = bet365["markets"].get("match_odds")

        if match_odds is None:
            missing_match_odds.append({
                "season": season_name,
                "match_id": details["id"],
                "home_team": details["home_team"]["name"],
                "away_team": details["away_team"]["name"]
            })

print("Matches missing match odds:", len(missing_match_odds))

In [ ]:
# Adding home/draw/away odds to our historical dataset.

odds_rows = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        match_id = match_data["details"]["data"]["id"]

        bookmakers = match_data["odds"]["data"]["bookmakers"]
        match_odds = bookmakers[0]["markets"]["match_odds"]

        odds_rows.append({
            "match_id": match_id,
            "home_odds": float(match_odds["home"]["last_seen"]),
            "draw_odds": float(match_odds["draw"]["last_seen"]),
            "away_odds": float(match_odds["away"]["last_seen"])
        })

odds_df = pd.DataFrame(odds_rows)

odds_df.shape

In [ ]:
# Merging into our DataFrame

historical_df = historical_df.merge(
    odds_df,
    on = "match_id",
    how = "left"
)

historical_df.shape

In [ ]:
historical_df[
    [
        "home_team",
        "away_team",
        "home_odds",
        "draw_odds",
        "away_odds"
    ]
].head()

In [ ]:
# Make sure none of the odds became missing after the merge.

historical_df[
    ["home_odds", "draw_odds", "away_odds"]
].isnull().sum()

In [ ]:
# Overwrite our historical matches CSV with updated DataFrame

historical_df.to_csv(
    historical_file,
    index = False 
)

print("Saved to:", historical_file)
print("Shape:", historical_df.shape)

# DATA PIPELINE PART 2
ADDING MORE COMPETITIONS

In [ ]:
# Set up Championship seasons and create a directory to store these matches

CHAMPIONSHIP_ID = "comp_8321"

CHAMPIONSHIP_SEASONS = {
    "2026_27": "sn_3014533",
    "2025_26": "sn_3064530",
    "2024_25": "sn_2930227",
    "2023_24": "sn_343481",
    "2022_23": "sn_286965",
    "2021_22": "sn_033091",
    "2020_21": "sn_855998"
}

CHAMPIONSHIP_DIR = RAW_MATCH_DIR / "championship"

CHAMPIONSHIP_SEASON_DIRS = {}

for season_name in CHAMPIONSHIP_SEASONS:
    season_dir = CHAMPIONSHIP_DIR / season_name
    season_dir.mkdir(parents = True, exist_ok = True)

    CHAMPIONSHIP_SEASON_DIRS[season_name] = season_dir

print(CHAMPIONSHIP_SEASON_DIRS)

In [ ]:
# Test on one Championship season
championship_2022_23_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2022_23"]
)

print("Total matches returned:", len(championship_2022_23_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2022_23_matches))

In [ ]:
# Check if the data is compatible

test_championship_match = championship_2022_23_matches[0]

print(
    test_championship_match["home_team"]["name"],
    "vs",
    test_championship_match["away_team"]["name"]
)

test_championship_data = get_baseline_match_data(
    test_championship_match["id"]
)

print(test_championship_data.keys())

In [ ]:
# Make a Championship save path
# Don't want to mix up the JSONs 

def collect_championship_baseline_data(season_name, matches):
    season_dir = CHAMPIONSHIP_SEASON_DIRS[season_name]

    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(
            match_id,
            match_data,
            season_dir
        )

        time.sleep(1)

In [ ]:
# Collect 2022/23 Championship matches

collect_championship_baseline_data(
    "2022_23",
    championship_2022_23_matches
)

In [ ]:
saved_championship_2021_22 = list(
    CHAMPIONSHIP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Championship 2021/22 files:", len(saved_championship_2021_22))

In [ ]:
championship_2020_21_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2020_21"]
)

print("Total matches returned:", len(championship_2020_21_matches))

In [ ]:
# Collect 2020_21 Championship season

collect_championship_baseline_data(
    "2020_21",
    championship_2020_21_matches
)

In [ ]:
saved_championship_2020_21 = list(
    CHAMPIONSHIP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved Championship 2020/21 files:", len(saved_championship_2020_21))

In [ ]:
# Collect 2023/24 season

championship_2023_24_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2023_24"]
)

print("Total matches returned:", len(championship_2023_24_matches))

In [ ]:
collect_championship_baseline_data(
    "2023_24",
    championship_2023_24_matches
)

In [ ]:
saved_championship_2023_24 = list(
    CHAMPIONSHIP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Championship 2023/24 files:", len(saved_championship_2023_24))

In [ ]:
# Collect 2024/25 Championship season
championship_2024_25_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2024_25"]
)

print("Total matches returned:", len(championship_2024_25_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in championship_2024_25_matches
)))

In [ ]:
finished_championship_2024_25 = [
    match for match in championship_2024_25_matches
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_championship_2024_25))

In [ ]:
collect_championship_baseline_data(
    "2024_25",
    finished_championship_2024_25
)

In [ ]:
saved_championship_2024_25 = list(
    CHAMPIONSHIP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Championship 2024/25 files:", len(saved_championship_2024_25))

In [ ]:
# Collect 2025/26 Championship season

championship_2025_26_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2025_26"]
)

print("Total matches returned:", len(championship_2025_26_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in championship_2025_26_matches
)))

In [ ]:
finished_championship_2025_26 = [
    match for match in championship_2025_26_matches
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_championship_2025_26))

In [ ]:
collect_championship_baseline_data(
    "2025_26",
    finished_championship_2025_26
)

In [ ]:
for season in [
    "2020_21",
    "2021_22",
    "2022_23",
    "2023_24",
    "2024_25",
    "2025_26"
]:
    files = list(CHAMPIONSHIP_SEASON_DIRS[season].glob("*.json"))
    print(season, len(files))

In [ ]:
# Adding Champions League Seasons

CHAMPIONS_LEAGUE_ID = "comp_3498"

CHAMPIONS_LEAGUE_SEASONS = {
    "2026_27": "sn_8406903",
    "2025_26": "sn_5783461",
    "2024_25": "sn_0872372",
    "2023_24": "sn_939193",
    "2022_23": "sn_327866",
    "2021_22": "sn_699943",
    "2020_21": "sn_388063"
}

In [ ]:
# Create a folder for Champions League matches

CHAMPIONS_LEAGUE_DIR = RAW_MATCH_DIR / "champions_league"

CHAMPIONS_LEAGUE_SEASON_DIRS = {}

for season_name in CHAMPIONS_LEAGUE_SEASONS:
    season_dir = CHAMPIONS_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)

    CHAMPIONS_LEAGUE_SEASON_DIRS[season_name] = season_dir

print(CHAMPIONS_LEAGUE_SEASON_DIRS)

In [ ]:
# Inspect 2023/24 season (Before format change)

ucl_2023_24_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(ucl_2023_24_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2023_24_matches
)))

In [ ]:
stage_examples = {}

for match in ucl_2023_24_matches:
    stage = match.get("stage_name")

    if stage not in stage_examples:
        stage_examples[stage] = match

for stage, match in stage_examples.items():
    print(
        stage,
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"]
    )

In [ ]:
# Check for Unique IDs

for stage in ["round_of_16", "semi_final", "final"]:
    print(f"\n--- {stage.upper()} ---")

    stage_matches = [
        match for match in ucl_2023_24_matches
        if match.get("stage_name") == stage
    ]

    for match in sorted(stage_matches, key=lambda x: x["utc_date"]):
        print(
            match["utc_date"][:10],
            match["id"],
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"]
        )

In [ ]:
# Create a collector for Champions League seasons

def collect_champions_league_baseline_data(season_name, matches):
    season_dir = CHAMPIONS_LEAGUE_SEASON_DIRS[season_name]

    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(
            match_id,
            match_data,
            season_dir
        )

        time.sleep(1)
 

In [ ]:
# Collect 2023_24 CL season

collect_champions_league_baseline_data(
    "2023_24",
    ucl_2023_24_matches
)

In [ ]:
saved_ucl_2023_24 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved UCL 2023/24 files:", len(saved_ucl_2023_24))

In [ ]:
# Collect 2022/23 CL season

ucl_2022_23_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(ucl_2022_23_matches))

In [ ]:
collect_champions_league_baseline_data(
    "2022_23",
    ucl_2022_23_matches
)

In [ ]:
saved_ucl_2022_23 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved UCL 2022/23 files:", len(saved_ucl_2022_23))

In [ ]:
# Collect 2021/22 CL season

ucl_2021_22_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(ucl_2021_22_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2021_22_matches
)))

In [ ]:
collect_champions_league_baseline_data(
    "2021_22",
    ucl_2021_22_matches
)

In [ ]:
saved_ucl_2021_22 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved UCL 2021/22 files:", len(saved_ucl_2021_22))

In [ ]:
# Collect 2020/21 CL season

ucl_2020_21_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2020_21"]
)

print("Total matches returned:", len(ucl_2020_21_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2020_21_matches
)))

In [ ]:
collect_champions_league_baseline_data(
    "2020_21",
    ucl_2020_21_matches
)

In [ ]:
saved_ucl_2020_21 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved UCL 2020/21 files:", len(saved_ucl_2020_21))

In [ ]:
# Collect 2024/25 CL season (new format)

ucl_2024_25_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(ucl_2024_25_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2024_25_matches
)))

In [ ]:
collect_champions_league_baseline_data(
    "2024_25",
    ucl_2024_25_matches
)

In [ ]:
saved_ucl_2024_25 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved UCL 2024/25 files:", len(saved_ucl_2024_25))

In [ ]:
# Collecting 2025/26 CL season

ucl_2025_26_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(ucl_2025_26_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2025_26_matches
)))

In [ ]:
collect_champions_league_baseline_data(
    "2025_26",
    ucl_2025_26_matches
)

In [ ]:
saved_ucl_2025_26 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved UCL 2025/26 files:", len(saved_ucl_2025_26))

In [ ]:
# Collecting FA cup seasons

FA_CUP_ID = "comp_7428"

FA_CUP_SEASONS = {
    "2026_27": "sn_7205142",
    "2025_26": "sn_3071665",
    "2024_25": "sn_8449488",
    "2023_24": "sn_885727",
    "2022_23": "sn_531226",
    "2021_22": "sn_097717",
    "2020_21": "sn_169084"
}

In [ ]:
# Create separate folder to store FA cup matches

FA_CUP_DIR = RAW_MATCH_DIR / "fa_cup"

FA_CUP_SEASON_DIRS = {}

for season_name in FA_CUP_SEASONS:
    season_dir = FA_CUP_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    FA_CUP_SEASON_DIRS[season_name] = season_dir

In [ ]:
# Collector for matches

def collect_fa_cup_baseline_data(season_name, matches):
    season_dir = FA_CUP_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [ ]:
# Collect 2023/24 FA Cup matches

fa_cup_2023_24_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2023_24"]
)

print("Total matches returned:", len(fa_cup_2023_24_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2023_24_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2023_24",
    fa_cup_2023_24_matches
)

In [ ]:
saved_fa_cup_2023_24 = list(
    FA_CUP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved FA Cup 2023/24 files:", len(saved_fa_cup_2023_24))

In [ ]:
# Collect 2022/23 FA cup season

fa_cup_2022_23_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2022_23"]
)

print("Total matches returned:", len(fa_cup_2022_23_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2022_23_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2022_23",
    fa_cup_2022_23_matches
)

In [ ]:
saved_fa_cup_2022_23 = list(
    FA_CUP_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved FA Cup 2022/23 files:", len(saved_fa_cup_2022_23))

In [ ]:
# Collect 2021/22 FA cup season

fa_cup_2021_22_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2021_22"]
)

print("Total matches returned:", len(fa_cup_2021_22_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2021_22_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2021_22",
    fa_cup_2021_22_matches
)

In [ ]:
saved_fa_cup_2021_22 = list(
    FA_CUP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved FA Cup 2021/22 files:", len(saved_fa_cup_2021_22))

In [ ]:
# Collect 2020/21 FA cup season

fa_cup_2020_21_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2020_21"]
)

print("Total matches returned:", len(fa_cup_2020_21_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2020_21_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2020_21",
    fa_cup_2020_21_matches
)

In [ ]:
saved_fa_cup_2020_21 = list(
    FA_CUP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved FA Cup 2020/21 files:", len(saved_fa_cup_2020_21))

In [ ]:
# Collect 2024/25 FA cup season

fa_cup_2024_25_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2024_25"]
)

print("Total matches returned:", len(fa_cup_2024_25_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2024_25_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2024_25",
    fa_cup_2024_25_matches
)

In [ ]:
saved_fa_cup_2024_25 = list(
    FA_CUP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved FA Cup 2024/25 files:", len(saved_fa_cup_2024_25))

In [ ]:
# Collect 2025/26 FA cup season

fa_cup_2025_26_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2025_26"]
)

print("Total matches returned:", len(fa_cup_2025_26_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2025_26_matches
)))

In [ ]:
collect_fa_cup_baseline_data(
    "2025_26",
    fa_cup_2025_26_matches
)

In [ ]:
saved_fa_cup_2025_26 = list(
    FA_CUP_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved FA Cup 2025/26 files:", len(saved_fa_cup_2025_26))

In [ ]:
# Creating code to collect EFL Cup matches

EFL_CUP_ID = "comp_2504"

EFL_CUP_SEASONS = {
    "2026_27": "sn_4588470",
    "2025_26": "sn_1398983",
    "2024_25": "sn_5762314",
    "2023_24": "sn_595992",
    "2022_23": "sn_532755",
    "2021_22": "sn_311334",
    "2020_21": "sn_044278"
}

In [ ]:
# Create separate folder for EFL Cup matches

EFL_CUP_DIR = RAW_MATCH_DIR / "efl_cup"

EFL_CUP_SEASON_DIRS = {}

for season_name in EFL_CUP_SEASONS:
    season_dir = EFL_CUP_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    EFL_CUP_SEASON_DIRS[season_name] = season_dir

In [ ]:
# Collector for matches in EFL Cup

def collect_efl_cup_baseline_data(season_name, matches):
    season_dir = EFL_CUP_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [ ]:
# Collect 2023/24 EFL Cup matches

efl_cup_2023_24_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2023_24"]
)

print("Total matches returned:", len(efl_cup_2023_24_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2023_24_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2023_24",
    efl_cup_2023_24_matches
)

In [ ]:
saved_efl_cup_2023_24 = list(
    EFL_CUP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved EFL Cup 2023/24 files:", len(saved_efl_cup_2023_24))

In [ ]:
# Collect 2022/23 EFL Cup season

efl_cup_2022_23_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2022_23"]
)

print("Total matches returned:", len(efl_cup_2022_23_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2022_23_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2022_23",
    efl_cup_2022_23_matches
)

In [ ]:
saved_efl_cup_2022_23 = list(
    EFL_CUP_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved EFL Cup 2022/23 files:", len(saved_efl_cup_2022_23))

In [ ]:
# Collect 2021/22 FA Cup season

efl_cup_2021_22_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2021_22"]
)

print("Total matches returned:", len(efl_cup_2021_22_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2021_22_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2021_22",
    efl_cup_2021_22_matches
)

In [ ]:
saved_efl_cup_2021_22 = list(
    EFL_CUP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved EFL Cup 2021/22 files:", len(saved_efl_cup_2021_22))

In [ ]:
# Collect 2020/21 FA Cup season

efl_cup_2020_21_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2020_21"]
)

print("Total matches returned:", len(efl_cup_2020_21_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2020_21_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2020_21",
    efl_cup_2020_21_matches
)

In [ ]:
saved_efl_cup_2020_21 = list(
    EFL_CUP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved EFL Cup 2020/21 files:", len(saved_efl_cup_2020_21))

In [ ]:
# Collect 2024/25 FA Cup season

efl_cup_2024_25_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2024_25"]
)

print("Total matches returned:", len(efl_cup_2024_25_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2024_25_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2024_25",
    efl_cup_2024_25_matches
)

In [ ]:
saved_efl_cup_2024_25 = list(
    EFL_CUP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved EFL Cup 2024/25 files:", len(saved_efl_cup_2024_25))

In [ ]:
# Collect 2025/26 FA Cup season

efl_cup_2025_26_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2025_26"]
)

print("Total matches returned:", len(efl_cup_2025_26_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2025_26_matches
)))

In [ ]:
collect_efl_cup_baseline_data(
    "2025_26",
    efl_cup_2025_26_matches
)

In [ ]:
saved_efl_cup_2025_26 = list(
    EFL_CUP_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved EFL Cup 2025/26 files:", len(saved_efl_cup_2025_26))

In [ ]:
# Create code to collect UEFA Europa League seasons

EUROPA_LEAGUE_ID = "comp_7739"

EUROPA_LEAGUE_SEASONS = {
    "2026_27": "sn_9624824",
    "2025_26": "sn_0819325",
    "2024_25": "sn_4596897",
    "2023_24": "sn_008323",
    "2022_23": "sn_706199",
    "2021_22": "sn_481111",
    "2020_21": "sn_522881"
}

In [ ]:
# Create a folder for UEFA Europa League matches

EUROPA_LEAGUE_DIR = RAW_MATCH_DIR / "europa_league"

EUROPA_LEAGUE_SEASON_DIRS = {}

for season_name in EUROPA_LEAGUE_SEASONS:
    season_dir = EUROPA_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    EUROPA_LEAGUE_SEASON_DIRS[season_name] = season_dir

In [ ]:
# Create the collector for Europa League seasons

def collect_europa_league_baseline_data(season_name, matches):
    season_dir = EUROPA_LEAGUE_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [ ]:
# Collect 2023/24 Europa League season

europa_2023_24_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(europa_2023_24_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2023_24_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2023_24",
    europa_2023_24_matches
)

In [ ]:
saved_europa_2023_24 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Europa League 2023/24 files:", len(saved_europa_2023_24))

In [ ]:
# Collect 2022/23 Europa League season

europa_2022_23_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(europa_2022_23_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2022_23_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2022_23",
    europa_2022_23_matches
)

In [ ]:
saved_europa_2022_23 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved Europa League 2022/23 files:", len(saved_europa_2022_23))

In [ ]:
# Collect 2021/22 Europa League season

europa_2021_22_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(europa_2021_22_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2021_22_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2021_22",
    europa_2021_22_matches
)

In [ ]:
saved_europa_2021_22 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Europa League 2021/22 files:", len(saved_europa_2021_22))

In [ ]:
# Collect 2020/21 Europa League season

europa_2020_21_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2020_21"]
)

print("Total matches returned:", len(europa_2020_21_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2020_21_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2020_21",
    europa_2020_21_matches
)

In [ ]:
saved_europa_2020_21 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved Europa League 2020/21 files:", len(saved_europa_2020_21))

In [ ]:
# Collecting 2024/25 Europa League season

europa_2024_25_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(europa_2024_25_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2024_25_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2024_25",
    europa_2024_25_matches
)

In [ ]:
saved_europa_2024_25 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Europa League 2024/25 files:", len(saved_europa_2024_25))

In [ ]:
# Collect 2025/26 Europa League season

europa_2025_26_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(europa_2025_26_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2025_26_matches
)))

In [ ]:
collect_europa_league_baseline_data(
    "2025_26",
    europa_2025_26_matches
)

In [ ]:
saved_europa_2025_26 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved Europa League 2025/26 files:", len(saved_europa_2025_26))

In [ ]:
# Set Up UEFA Conference League seasons

CONFERENCE_LEAGUE_ID = "comp_408698"

CONFERENCE_LEAGUE_SEASONS = {
    "2026_27": "sn_7250309",
    "2025_26": "sn_1397633",
    "2024_25": "sn_8429027",
    "2023_24": "sn_343450",
    "2022_23": "sn_092013",
    "2021_22": "sn_033592"
}

In [ ]:
# Create folder for UEFA Conference League matches

CONFERENCE_LEAGUE_DIR = RAW_MATCH_DIR / "conference_league"

CONFERENCE_LEAGUE_SEASON_DIRS = {}

for season_name in CONFERENCE_LEAGUE_SEASONS:
    season_dir = CONFERENCE_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    CONFERENCE_LEAGUE_SEASON_DIRS[season_name] = season_dir

In [ ]:
# Create a collector for UEFA Conference League seasons

def collect_conference_league_baseline_data(season_name, matches):
    season_dir = CONFERENCE_LEAGUE_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)

        time.sleep(1)

In [ ]:
# Collect 2021/22 Conference League season

conference_2021_22_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(conference_2021_22_matches))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2021_22_matches
)))

In [ ]:
conference_2021_22_filtered = [
    match
    for match in conference_2021_22_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2021_22_matches))
print("Matches after filtering:", len(conference_2021_22_filtered))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2021_22_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2021_22_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2021_22_filtered
)))

In [ ]:
collect_conference_league_baseline_data(
    "2021_22",
    conference_2021_22_filtered
)

In [ ]:
saved_conference_2021_22 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Conference League 2021/22 files:", len(saved_conference_2021_22))

In [ ]:
# Collect 2022/23 Conference League season

conference_2022_23_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(conference_2022_23_matches))

In [ ]:
conference_2022_23_filtered = [
    match
    for match in conference_2022_23_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2022_23_matches))
print("Matches after filtering:", len(conference_2022_23_filtered))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2022_23_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2022_23_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2022_23_filtered
)))

In [ ]:
collect_conference_league_baseline_data(
    "2022_23",
    conference_2022_23_filtered
)

In [ ]:
saved_conference_2022_23 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved Conference League 2022/23 files:", len(saved_conference_2022_23))

In [ ]:
# Collect 2023/24 Conference League season

conference_2023_24_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(conference_2023_24_matches))

In [ ]:
conference_2023_24_filtered = [
    match
    for match in conference_2023_24_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2023_24_matches))
print("Matches after filtering:", len(conference_2023_24_filtered))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2023_24_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2023_24_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2023_24_filtered
)))

In [ ]:
collect_conference_league_baseline_data(
    "2023_24",
    conference_2023_24_filtered
)

In [ ]:
saved_conference_2023_24 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Conference League 2023/24 files:", len(saved_conference_2023_24))

In [ ]:
# Collect 2024/25 Europa League season

conference_2024_25_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(conference_2024_25_matches))

In [ ]:
conference_2024_25_filtered = [
    match
    for match in conference_2024_25_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2024_25_matches))
print("Matches after filtering:", len(conference_2024_25_filtered))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2024_25_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2024_25_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2024_25_filtered
)))

In [ ]:
collect_conference_league_baseline_data(
    "2024_25",
    conference_2024_25_filtered
)

In [ ]:
saved_conference_2024_25 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Conference League 2024/25 files:", len(saved_conference_2024_25))

In [ ]:
# Collect 2025/26 Conference League season

conference_2025_26_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(conference_2025_26_matches))

In [ ]:
conference_2025_26_filtered = [
    match
    for match in conference_2025_26_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2025_26_matches))
print("Matches after filtering:", len(conference_2025_26_filtered))

In [ ]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2025_26_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2025_26_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2025_26_filtered
)))

In [ ]:
collect_conference_league_baseline_data(
    "2025_26",
    conference_2025_26_filtered
)

In [ ]:
saved_conference_2025_26 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved Conference League 2025/26 files:", len(saved_conference_2025_26))

In [ ]:
PREMIER_LEAGUE_DIR = RAW_MATCH_DIR / "premier_league"

SEASON_DIRS = {}

for season_name in SEASONS:
    season_dir = PREMIER_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    SEASON_DIRS[season_name] = season_dir

In [ ]:
for season_name, season_dir in SEASON_DIRS.items():
    print(season_name, "->", season_dir)

In [ ]:
# Updating get_stat for the matches with no available statistics

def get_stat(stats, stat_name, side):
    if stats is None:
        return None

    stats = stats.get("data")

    if stats is None:
        return None
    
    overview = stats.get("overview")

    if overview is None:
        return None

    stat = overview.get(stat_name)

    if stat is None or stat.get("all") is None:
        return None

    return stat["all"].get(side)

In [ ]:
# Update historical match extractor so every processed row includes both season and competition 

def extract_historical_match(match_data, season, competition, match_id):
    details = match_data["details"]["data"]
    stats = match_data.get("statistics")

    return {
        "match_id": match_id,
        "date": details["utc_date"],
        "season": season,
        "competition": competition,
        "matchday": details.get("matchday"),

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": get_stat(stats, "expected_goals", "home"),
        "away_xg": get_stat(stats, "expected_goals", "away"),

        "home_shots": get_stat(stats, "total_shots", "home"),
        "away_shots": get_stat(stats, "total_shots", "away"),

        "home_shots_on_target": get_stat(stats, "shots_on_target", "home"),
        "away_shots_on_target": get_stat(stats, "shots_on_target", "away"),

        "home_possession": get_stat(stats, "ball_possession", "home"),
        "away_possession": get_stat(stats, "ball_possession", "away"),

        "home_passes": get_stat(stats, "passes", "home"),
        "away_passes": get_stat(stats, "passes", "away"),

        "home_corners": get_stat(stats, "corner_kicks", "home"),
        "away_corners": get_stat(stats, "corner_kicks", "away"),

        "winner": details["score"].get("winner")
    }

In [ ]:
# Create a map telling Python which folder belongs to which competition.

COMPETITION_DIRS = {
    "premier_league": RAW_MATCH_DIR / "premier_league",
    "championship": RAW_MATCH_DIR / "championship",
    "champions_league": RAW_MATCH_DIR / "champions_league",
    "fa_cup": RAW_MATCH_DIR / "fa_cup",
    "efl_cup": RAW_MATCH_DIR / "efl_cup",
    "europa_league": RAW_MATCH_DIR / "europa_league",
    "conference_league": RAW_MATCH_DIR / "conference_league"
}

In [ ]:
for competition, directory in COMPETITION_DIRS.items():
    print(competition, "->", directory.exists(), "->", directory)

In [ ]:
# Separate current season (2026/27) from historical seasons

COMPLETED_SEASONS = {
    "premier_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "championship": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "champions_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "fa_cup": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "efl_cup": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "europa_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "conference_league": ["2021_22", "2022_23", "2023_24", "2024_25", "2025_26"]
}

for competition, seasons in COMPLETED_SEASONS.items():
    print(competition, "->", len(seasons), "seasons")

In [ ]:
# Build loop that reads every historical JSON file and sends it through the extractor

all_historical_matches = []

for competition, seasons in COMPLETED_SEASONS.items():
    competition_dir = COMPETITION_DIRS[competition]

    for season in seasons:
        season_dir = competition_dir / season

        for file_path in season_dir.glob("*.json"):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    match_data = json.load(f)

                match_id = file_path.stem

                match_row = extract_historical_match(
                    match_data,
                    season,
                    competition,
                    match_id
                )

                all_historical_matches.append(match_row)

            except Exception as e:
                print("Error processing:", file_path)
                print("Error:", e)

print("Total completed historical matches processed:", len(all_historical_matches))

In [ ]:
# Turn the list into a DataFrame

historical_df = pd.DataFrame(all_historical_matches)

print(historical_df.shape)
print()
print(historical_df.head())
print()
print(historical_df["competition"].value_counts())

In [ ]:
# Check which columns are complete and which ones are sparse before saving into CSV

missing_summary = historical_df.isna().sum().sort_values(ascending=False)

print(missing_summary)

In [ ]:
historical_df.to_csv(
    "../data/processed/historical_matches.csv",
    index=False
)

print("Saved:", historical_df.shape)

In [ ]:
import json
from pathlib import Path

sample_file = next(
    (
        Path("../data/raw/matches/premier_league/2024_25")
    ).glob("*.json")
)

with open(
    sample_file,
    "r",
    encoding="utf-8"
) as f:
    sample_match = json.load(f)

print("Match file:", sample_file.name)
print()
print(
    json.dumps(
        sample_match.get("odds"),
        indent=2
    )[:5000]
)

In [ ]:
def extract_bet365_match_odds(match_data, match_id):

    odds_data = match_data.get("odds")

    if odds_data is None:
        return None

    odds_data = odds_data.get("data")

    if odds_data is None:
        return None

    bookmakers = odds_data.get("bookmakers", [])

    bet365 = None

    for bookmaker in bookmakers:
        if bookmaker.get("bookmaker") == "Bet365":
            bet365 = bookmaker
            break

    if bet365 is None:
        return None

    match_odds = (
        bet365
        .get("markets", {})
        .get("match_odds")
    )

    if match_odds is None:
        return None

    try:
        home_odds = float(
            match_odds["home"]["last_seen"]
        )

        draw_odds = float(
            match_odds["draw"]["last_seen"]
        )

        away_odds = float(
            match_odds["away"]["last_seen"]
        )

    except (TypeError, ValueError, KeyError):
        return None

    return {
        "match_id": match_id,
        "bet365_home_odds": home_odds,
        "bet365_draw_odds": draw_odds,
        "bet365_away_odds": away_odds
    }

In [ ]:
sample_odds = extract_bet365_match_odds(
    sample_match,
    sample_file.stem
)

print(sample_odds)

In [ ]:
# See how much Bet365 coverage we have across 6 completed PL seasons

bet365_odds_rows = []

for season in COMPLETED_SEASONS["premier_league"]:

    season_dir = (
        COMPETITION_DIRS["premier_league"]
        / season
    )

    season_total = 0
    season_with_odds = 0

    for file_path in season_dir.glob("*.json"):

        season_total += 1

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:
            match_data = json.load(f)

        odds_row = extract_bet365_match_odds(
            match_data,
            file_path.stem
        )

        if odds_row is not None:

            odds_row["season"] = season

            bet365_odds_rows.append(
                odds_row
            )

            season_with_odds += 1

    print(
        f"{season}: "
        f"{season_with_odds}/{season_total} "
        f"matches with Bet365 odds"
    )

print()
print(
    "Total Bet365 matches:",
    len(bet365_odds_rows)
)

In [ ]:
# Turn these 2,280 rows into a DataFrame and calculate fair market probabilities by removing Bet365 margin

bet365_df = pd.DataFrame(
    bet365_odds_rows
)

# Raw implied probabilities
bet365_df["raw_prob_H"] = (
    1 / bet365_df["bet365_home_odds"]
)

bet365_df["raw_prob_D"] = (
    1 / bet365_df["bet365_draw_odds"]
)

bet365_df["raw_prob_A"] = (
    1 / bet365_df["bet365_away_odds"]
)

# Bookmaker overround
bet365_df["overround"] = (
    bet365_df["raw_prob_H"]
    + bet365_df["raw_prob_D"]
    + bet365_df["raw_prob_A"]
)

# Normalize to remove bookmaker margin
bet365_df["fair_prob_H"] = (
    bet365_df["raw_prob_H"]
    / bet365_df["overround"]
)

bet365_df["fair_prob_D"] = (
    bet365_df["raw_prob_D"]
    / bet365_df["overround"]
)

bet365_df["fair_prob_A"] = (
    bet365_df["raw_prob_A"]
    / bet365_df["overround"]
)

print(bet365_df.shape)

print()

print(
    bet365_df[
        [
            "bet365_home_odds",
            "bet365_draw_odds",
            "bet365_away_odds",
            "overround",
            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A"
        ]
    ].head()
)

print()

print(
    "Average overround:",
    bet365_df["overround"].mean()
)

print(
    "Average fair probability sum:",
    (
        bet365_df["fair_prob_H"]
        + bet365_df["fair_prob_D"]
        + bet365_df["fair_prob_A"]
    ).mean()
)

In [ ]:
# Save Bet365 data
bet365_df.to_csv(
    "../data/processed/bet365_pl_odds.csv",
    index=False
)

print(
    "Saved:",
    bet365_df.shape
)

In [ ]:
# ---------------------------------------------------------
# PROCESS ONE RAW MATCH FILE
# ---------------------------------------------------------

def process_match_file(file_path):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # Only completed matches
    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "statistics"
    )


    # -----------------------------------------------------
    # TEAM INFORMATION
    # -----------------------------------------------------

    home_team_data = details.get(
        "home_team",
        {}
    )

    away_team_data = details.get(
        "away_team",
        {}
    )


    home_team = (
        home_team_data.get("name")
        if isinstance(home_team_data, dict)
        else None
    )

    away_team = (
        away_team_data.get("name")
        if isinstance(away_team_data, dict)
        else None
    )


    # -----------------------------------------------------
    # SCORE
    # -----------------------------------------------------

    score = details.get(
        "score",
        {}
    )

    if not isinstance(score, dict):
        score = {}


    home_goals = score.get(
        "home"
    )

    away_goals = score.get(
        "away"
    )


    # -----------------------------------------------------
    # WINNER
    # -----------------------------------------------------

    winner = None

    if (
        home_goals is not None
        and away_goals is not None
    ):

        if home_goals > away_goals:

            winner = "H"

        elif away_goals > home_goals:

            winner = "A"

        else:

            winner = "D"


    # -----------------------------------------------------
    # FOLDER INFORMATION
    # -----------------------------------------------------

    season = file_path.parent.name

    competition = (
        file_path
        .parent
        .parent
        .name
    )


    # -----------------------------------------------------
    # BUILD ROW
    # -----------------------------------------------------

    row = {

        "match_id":
            details.get("id"),

        "date":
            details.get("utc_date"),

        "season":
            season,

        "competition":
            competition,

        "matchday":
            details.get("matchday"),

        "home_team":
            home_team,

        "away_team":
            away_team,

        "home_goals":
            home_goals,

        "away_goals":
            away_goals,

        "home_xg":
            get_stat(
                statistics,
                "home",
                "expected_goals"
            ),

        "away_xg":
            get_stat(
                statistics,
                "away",
                "expected_goals"
            ),

        "home_shots":
            get_stat(
                statistics,
                "home",
                "shots"
            ),

        "away_shots":
            get_stat(
                statistics,
                "away",
                "shots"
            ),

        "home_shots_on_target":
            get_stat(
                statistics,
                "home",
                "shots_on_target"
            ),

        "away_shots_on_target":
            get_stat(
                statistics,
                "away",
                "shots_on_target"
            ),

        "home_possession":
            get_stat(
                statistics,
                "home",
                "ball_possession"
            ),

        "away_possession":
            get_stat(
                statistics,
                "away",
                "ball_possession"
            ),

        "home_passes":
            get_stat(
                statistics,
                "home",
                "passes"
            ),

        "away_passes":
            get_stat(
                statistics,
                "away",
                "passes"
            ),

        "home_corners":
            get_stat(
                statistics,
                "home",
                "corners"
            ),

        "away_corners":
            get_stat(
                statistics,
                "away",
                "corners"
            ),

        "winner":
            winner
    }


    return row

In [ ]:
# ---------------------------------------------------------
# CHECK 2026/27 PROCESSING
# ---------------------------------------------------------

check_2026 = historical_df[
    (
        historical_df["competition"] == "premier_league"
    )
    &
    (
        historical_df["season"] == "2026_27"
    )
][
    [
        "date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "winner"
    ]
].sort_values("date")


print(
    check_2026
)

print()

print(
    "Missing dates:",
    check_2026["date"].isna().sum()
)

print(
    "Missing home goals:",
    check_2026["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    check_2026["away_goals"].isna().sum()
)

In [ ]:
import inspect

print(
    inspect.getsource(
        process_match_file
    )
)

In [ ]:
# ---------------------------------------------------------
# INSPECT ONE 2026/27 RAW MATCH
# ---------------------------------------------------------

import json
from pathlib import Path
from pprint import pprint


season_dir = Path(
    "../data/raw/matches/premier_league/2026_27"
)


match_files = list(
    season_dir.glob("*.json")
)


print(
    "2026/27 raw files:",
    len(match_files)
)


with open(
    match_files[0],
    "r",
    encoding="utf-8"
) as f:

    sample_match = json.load(f)


print("\nTOP-LEVEL KEYS:")
print(
    sample_match.keys()
)


print("\nDETAILS:")
pprint(
    sample_match.get("details")
)

In [ ]:
# ---------------------------------------------------------
# TEST PROCESSOR ON ONE 2026/27 RAW MATCH
# ---------------------------------------------------------

sample_file = match_files[0]

print(
    "FILE:",
    sample_file
)

print()

processed_sample = process_match_file(
    sample_file
)

print(
    processed_sample
)

In [ ]:
# ---------------------------------------------------------
# PROCESS ALL 20 2026/27 FILES DIRECTLY
# ---------------------------------------------------------

test_2026_rows = []

for file_path in sorted(
    Path("../data/raw/matches/premier_league/2026_27").glob("*.json")
):

    row = process_match_file(
        file_path
    )

    if row is not None:
        test_2026_rows.append(row)


test_2026_df = pd.DataFrame(
    test_2026_rows
)


print(
    "Processed matches:",
    len(test_2026_df)
)

print()

print(
    test_2026_df[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "winner"
        ]
    ]
    .sort_values("date")
    .to_string(index=False)
)

print()

print(
    "Missing dates:",
    test_2026_df["date"].isna().sum()
)

print(
    "Missing home goals:",
    test_2026_df["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    test_2026_df["away_goals"].isna().sum()
)

print(
    "Missing winners:",
    test_2026_df["winner"].isna().sum()
)

In [ ]:
# ---------------------------------------------------------
# CLEAN REBUILD OF HISTORICAL DATASET
# ---------------------------------------------------------

rows = []


# Process every raw match again from scratch
for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


# Create completely new dataframe
historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATES
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# CONVERT AND SORT DATES
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# SAVE FRESH CSV
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# VERIFY 2026/27
# ---------------------------------------------------------

check_2026 = (
    historical_df[
        (
            historical_df["competition"]
            == "premier_league"
        )
        &
        (
            historical_df["season"]
            == "2026_27"
        )
    ][
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "winner"
        ]
    ]
    .sort_values("date")
)


print(
    "Historical shape:",
    historical_df.shape
)

print()

print(
    "2026/27 PL matches:",
    len(check_2026)
)

print()

print(
    check_2026.to_string(
        index=False
    )
)

print()

print(
    "Missing dates:",
    check_2026["date"].isna().sum()
)

print(
    "Missing home goals:",
    check_2026["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    check_2026["away_goals"].isna().sum()
)

print(
    "Missing winners:",
    check_2026["winner"].isna().sum()
)

print()

print(
    "Saved:",
    output_path
)

In [ ]:
# ---------------------------------------------------------
# INSPECT RAW STATISTICS STRUCTURE
# ---------------------------------------------------------

from pprint import pprint
import json
from pathlib import Path


sample_file = Path(
    "../data/raw/matches/premier_league/2026_27/mt_022587821.json"
)


with open(
    sample_file,
    "r",
    encoding="utf-8"
) as f:

    sample_match = json.load(f)


print("STATISTICS:")
print()

pprint(
    sample_match.get("statistics")
)

In [ ]:
# ---------------------------------------------------------
# SAFE STAT EXTRACTOR
# ---------------------------------------------------------

def get_stat(statistics, team_side, stat_name):

    if statistics is None:
        return None


    # Unwrap API "data"
    if isinstance(statistics, dict):

        statistics = statistics.get(
            "data",
            statistics
        )


    if not isinstance(statistics, dict):
        return None


    # -----------------------------------------------------
    # MAP OUR DATASET NAMES TO API STAT NAMES
    # -----------------------------------------------------

    stat_name_map = {

        "expected_goals":
            "expected_goals",

        "shots":
            "total_shots",

        "shots_on_target":
            "shots_on_target",

        "ball_possession":
            "ball_possession",

        "passes":
            "passes",

        "corners":
            "corner_kicks"
    }


    api_stat_name = stat_name_map.get(
        stat_name,
        stat_name
    )


    # -----------------------------------------------------
    # NEW API STRUCTURE:
    # data -> overview -> stat -> all -> home/away
    # -----------------------------------------------------

    overview = statistics.get(
        "overview",
        {}
    )


    if isinstance(overview, dict):

        stat_data = overview.get(
            api_stat_name
        )


        if isinstance(stat_data, dict):

            all_data = stat_data.get(
                "all"
            )


            if isinstance(all_data, dict):

                value = all_data.get(
                    team_side
                )


                if value is not None:
                    return value


    # -----------------------------------------------------
    # FALLBACK FOR OLDER RAW FILE STRUCTURES
    # -----------------------------------------------------

    team_stats = statistics.get(
        team_side,
        {}
    )


    if isinstance(team_stats, dict):

        value = team_stats.get(
            stat_name
        )


        if value is not None:
            return value


    return None

In [ ]:
# ---------------------------------------------------------
# TEST UPDATED STAT EXTRACTION
# ---------------------------------------------------------

processed_sample = process_match_file(
    sample_file
)


for column in [
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]:

    print(
        column,
        processed_sample[column]
    )

In [ ]:
# ---------------------------------------------------------
# FINAL CLEAN REBUILD WITH CORRECT STAT EXTRACTION
# ---------------------------------------------------------

rows = []


for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATES
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# DATE + SORT
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# CHECK 2026/27 STAT COVERAGE
# ---------------------------------------------------------

check_2026 = historical_df[
    (
        historical_df["competition"]
        == "premier_league"
    )
    &
    (
        historical_df["season"]
        == "2026_27"
    )
].copy()


print(
    "Historical shape:",
    historical_df.shape
)

print(
    "2026/27 PL matches:",
    len(check_2026)
)

print()

print("2026/27 MISSING VALUES")

for column in [
    "date",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]:

    print(
        f"{column}:",
        check_2026[column].isna().sum()
    )


print()

print(
    check_2026[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "home_shots_on_target",
            "away_shots_on_target"
        ]
    ]
    .sort_values("date")
    .to_string(index=False)
)


print()

print(
    "Saved:",
    output_path
)

# VERSION 2

In [ ]:
# =========================================================
# HISTORICAL DATA PIPELINE V2
# Corrected raw JSON extraction
# =========================================================

from pathlib import Path
import json
import pandas as pd


# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------

RAW_MATCH_DIR = Path("../data/raw/matches")
PROCESSED_DIR = Path("../data/processed")


# ---------------------------------------------------------
# STAT NAME MAPPING
#
# Maps our preferred dataframe column terminology
# to the actual names used in the raw JSON.
# ---------------------------------------------------------

STAT_NAME_MAP = {
    "expected_goals": "expected_goals",
    "shots": "total_shots",
    "shots_on_target": "shots_on_target",
    "ball_possession": "ball_possession",
    "passes": "passes",
    "corners": "corner_kicks",
}


# ---------------------------------------------------------
# GET ONE STAT
#
# Actual JSON structure:
#
# statistics
#   -> data
#       -> overview
#           -> statistic
#               -> all
#                   -> home / away
# ---------------------------------------------------------

def get_stat_v2(statistics, team_side, stat_name):

    if statistics is None:
        return None

    # Unwrap "data"
    if isinstance(statistics, dict):
        statistics = statistics.get(
            "data",
            statistics
        )

    if not isinstance(statistics, dict):
        return None

    # Statistics are stored inside "overview"
    overview = statistics.get(
        "overview",
        {}
    )

    if not isinstance(overview, dict):
        return None

    # Convert our stat name to the raw API stat name
    api_stat_name = STAT_NAME_MAP.get(
        stat_name,
        stat_name
    )

    stat_data = overview.get(
        api_stat_name
    )

    if not isinstance(stat_data, dict):
        return None

    # Use full-match statistics
    all_data = stat_data.get(
        "all"
    )

    if not isinstance(all_data, dict):
        return None

    return all_data.get(
        team_side
    )


# ---------------------------------------------------------
# PROCESS ONE RAW MATCH FILE
# ---------------------------------------------------------

def process_match_file_v2(file_path):

    # -----------------------------------------------------
    # LOAD JSON
    # -----------------------------------------------------

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    # Unwrap API "data" object
    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # -----------------------------------------------------
    # ONLY COMPLETED MATCHES
    # -----------------------------------------------------

    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "statistics"
    )


    # -----------------------------------------------------
    # TEAM INFORMATION
    # -----------------------------------------------------

    home_team_data = details.get(
        "home_team",
        {}
    )

    away_team_data = details.get(
        "away_team",
        {}
    )


    home_team = (
        home_team_data.get("name")
        if isinstance(home_team_data, dict)
        else None
    )

    away_team = (
        away_team_data.get("name")
        if isinstance(away_team_data, dict)
        else None
    )


    # -----------------------------------------------------
    # SCORE
    #
    # Actual structure:
    #
    # details["score"]["home"]
    # details["score"]["away"]
    # -----------------------------------------------------

    score = details.get(
        "score",
        {}
    )

    if isinstance(score, dict):

        home_goals = score.get(
            "home"
        )

        away_goals = score.get(
            "away"
        )

    else:

        home_goals = None
        away_goals = None


    # -----------------------------------------------------
    # WINNER
    # -----------------------------------------------------

    winner = None

    if (
        home_goals is not None
        and away_goals is not None
    ):

        try:

            if home_goals > away_goals:
                winner = "H"

            elif away_goals > home_goals:
                winner = "A"

            else:
                winner = "D"

        except TypeError:
            winner = None


    # -----------------------------------------------------
    # FOLDER INFORMATION
    #
    # Structure:
    #
    # data/raw/matches/
    #     competition/
    #         season/
    #             match.json
    # -----------------------------------------------------

    season = file_path.parent.name

    competition = (
        file_path
        .parent
        .parent
        .name
    )


    # -----------------------------------------------------
    # BUILD ROW
    # -----------------------------------------------------

    row = {

        "match_id":
            details.get("id"),

        # Correct raw JSON field is utc_date
        "date":
            details.get("utc_date"),

        "season":
            season,

        "competition":
            competition,

        "matchday":
            details.get("matchday"),

        "home_team":
            home_team,

        "away_team":
            away_team,

        "home_goals":
            home_goals,

        "away_goals":
            away_goals,


        # -------------------------------------------------
        # EXPECTED GOALS
        # -------------------------------------------------

        "home_xg":
            get_stat_v2(
                statistics,
                "home",
                "expected_goals"
            ),

        "away_xg":
            get_stat_v2(
                statistics,
                "away",
                "expected_goals"
            ),


        # -------------------------------------------------
        # TOTAL SHOTS
        # -------------------------------------------------

        "home_shots":
            get_stat_v2(
                statistics,
                "home",
                "shots"
            ),

        "away_shots":
            get_stat_v2(
                statistics,
                "away",
                "shots"
            ),


        # -------------------------------------------------
        # SHOTS ON TARGET
        # -------------------------------------------------

        "home_shots_on_target":
            get_stat_v2(
                statistics,
                "home",
                "shots_on_target"
            ),

        "away_shots_on_target":
            get_stat_v2(
                statistics,
                "away",
                "shots_on_target"
            ),


        # -------------------------------------------------
        # POSSESSION
        # -------------------------------------------------

        "home_possession":
            get_stat_v2(
                statistics,
                "home",
                "ball_possession"
            ),

        "away_possession":
            get_stat_v2(
                statistics,
                "away",
                "ball_possession"
            ),


        # -------------------------------------------------
        # PASSES
        # -------------------------------------------------

        "home_passes":
            get_stat_v2(
                statistics,
                "home",
                "passes"
            ),

        "away_passes":
            get_stat_v2(
                statistics,
                "away",
                "passes"
            ),


        # -------------------------------------------------
        # CORNERS
        # -------------------------------------------------

        "home_corners":
            get_stat_v2(
                statistics,
                "home",
                "corners"
            ),

        "away_corners":
            get_stat_v2(
                statistics,
                "away",
                "corners"
            ),


        # -------------------------------------------------
        # TARGET
        # -------------------------------------------------

        "winner":
            winner
    }


    return row


# ---------------------------------------------------------
# PROCESS ALL RAW JSON FILES
# ---------------------------------------------------------

rows_v2 = []


for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file_v2(
        file_path
    )

    if row is not None:
        rows_v2.append(row)


historical_df_v2 = pd.DataFrame(
    rows_v2
)


# ---------------------------------------------------------
# REMOVE DUPLICATE MATCH IDS
# ---------------------------------------------------------

historical_df_v2 = (
    historical_df_v2
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# CONVERT DATE
# ---------------------------------------------------------

historical_df_v2["date"] = pd.to_datetime(
    historical_df_v2["date"],
    utc=True,
    errors="coerce"
)


# ---------------------------------------------------------
# SORT CHRONOLOGICALLY
# ---------------------------------------------------------

historical_df_v2 = (
    historical_df_v2
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# SAVE
#
# IMPORTANT:
# Save as V2 so the existing working dataset is untouched.
# ---------------------------------------------------------

output_path_v2 = (
    PROCESSED_DIR
    / "historical_matches_v2.csv"
)


historical_df_v2.to_csv(
    output_path_v2,
    index=False
)


# =========================================================
# VALIDATION
# =========================================================

print(
    "Historical V2 shape:",
    historical_df_v2.shape
)

print()

print(
    "COMPETITION COUNTS"
)

print()

print(
    historical_df_v2[
        "competition"
    ]
    .value_counts()
)


# ---------------------------------------------------------
# PREMIER LEAGUE
# ---------------------------------------------------------

pl_v2 = historical_df_v2[
    historical_df_v2[
        "competition"
    ]
    == "premier_league"
].copy()


print()
print(
    "PREMIER LEAGUE MATCHES:",
    len(pl_v2)
)


print()
print(
    "PREMIER LEAGUE BY SEASON"
)

print()

print(
    pl_v2[
        "season"
    ]
    .value_counts()
    .sort_index()
)


# ---------------------------------------------------------
# DATE RANGE
# ---------------------------------------------------------

print()
print(
    "DATE RANGE"
)

print()

print(
    pl_v2["date"].min(),
    "→",
    pl_v2["date"].max()
)


# ---------------------------------------------------------
# CORE FIELD VALIDATION
# ---------------------------------------------------------

print()
print(
    "MISSING CORE FIELDS"
)

print()

core_cols = [
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "winner",
]

print(
    pl_v2[
        core_cols
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# STAT COVERAGE
# ---------------------------------------------------------

print()
print(
    "PREMIER LEAGUE STAT COVERAGE"
)

print()

stat_cols = [
    "home_xg",
    "home_shots",
    "home_shots_on_target",
    "home_possession",
    "home_passes",
    "home_corners",
]

coverage = (
    pl_v2
    .groupby("season")[
        stat_cols
    ]
    .count()
)

print(
    coverage.to_string()
)


# ---------------------------------------------------------
# SAMPLE MATCHES
# ---------------------------------------------------------

print()
print(
    "SAMPLE PREMIER LEAGUE MATCHES"
)

print()

sample_cols = [
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "winner",
]

print(
    pl_v2[
        sample_cols
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# SAVE CONFIRMATION
# ---------------------------------------------------------

print()
print(
    "Saved:",
    output_path_v2
)

# CAPTURING PLAYERS' HEADSHOTS

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)

api_football_key = os.getenv("API_FOOTBALL_KEY")

url = "https://v3.football.api-sports.io/players"

params = {
    "search": "Saka",
    "league": 39,
    "season": 2024
}

headers_api_football = {
    "x-apisports-key": api_football_key
}

response = requests.get(
    url,
    headers=headers_api_football,
    params=params,
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    result = response.json()

    print("API errors:", result.get("errors"))
    print("Players returned:", result.get("results"))

    for item in result.get("response", [])[:5]:
        player = item.get("player", {})

        print(
            player.get("id"),
            "|",
            player.get("name"),
            "|",
            player.get("photo")
        )
else:
    print(response.text[:300])

In [ ]:
# Identify unique players appearing in our statistical leaderboards

leaderboard_columns = [
    "shooting_goals",
    "passing_assists",
    "shooting_expected_goals",
    "shooting_expected_assists",
    "shooting_shots_on_target",
    "shooting_total_shots",
    "shooting_big_chances_created",
    "passing_key_passes",
    "passing_accurate_passes",
    "defending_tackles",
    "defending_interceptions",
    "defending_clearances",
    "defending_ball_recoveries",
    "general_yellow_cards",
    "goalkeeping_saves",
    "minutes_played"
]

# Only include players who appeared in matches
active_players = player_df[
    player_df["minutes_played"].fillna(0) > 0
].copy()

# Aggregate season statistics
leaderboard_summary = (
    active_players.groupby(
        ["player_id", "player_name"],
        as_index=False
    )[leaderboard_columns]
    .sum()
)

# Collect the top five players from each leaderboard
top_player_ids = set()

for column in leaderboard_columns:
    top_five = leaderboard_summary.nlargest(5, column)
    top_player_ids.update(top_five["player_id"])

leaderboard_players = (
    leaderboard_summary[
        leaderboard_summary["player_id"].isin(top_player_ids)
    ][["player_id", "player_name"]]
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("Unique leaderboard players:", len(leaderboard_players))
print(leaderboard_players.to_string(index=False))

In [ ]:
from pathlib import Path

# Create a working copy of the 57 leaderboard players
photo_candidates = leaderboard_players.copy()

# Extract the last name to use for API-Football searches
photo_candidates["search_name"] = (
    photo_candidates["player_name"]
    .str.strip()
    .str.split()
    .str[-1]
)

# Prepare fields for the matching process
photo_candidates["api_football_id"] = pd.NA
photo_candidates["api_football_name"] = pd.NA
photo_candidates["photo_url"] = pd.NA
photo_candidates["match_status"] = "pending"

print("Players to match:", len(photo_candidates))

print(
    photo_candidates[
        ["player_name", "search_name", "match_status"]
    ].head(15).to_string(index=False)
)

In [ ]:
import requests
import time

test_names = [
    "Truffert",
    "Iwobi",
    "Semenyo",
    "Silva",
    "Fernandes"
]

for name in test_names:
    response = requests.get(
        "https://v3.football.api-sports.io/players",
        headers=headers_api_football,
        params={
            "search": name,
            "league": 39,
            "season": 2024
        },
        timeout=30
    )

    print(f"\n--- {name} ---")
    print("HTTP:", response.status_code)

    if response.status_code != 200:
        print("Request failed")
        break

    result = response.json()

    if result.get("errors"):
        print("API errors:", result["errors"])
        break

    print("Results:", result.get("results", 0))
    print("Pages:", result.get("paging", {}).get("total", 1))

    for item in result.get("response", [])[:10]:
        p = item["player"]

        print(
            p.get("id"),
            "|", p.get("name"),
            "|", p.get("birth", {}).get("date"),
            "|", p.get("nationality"),
            "|", p.get("photo")
        )

    time.sleep(2)

In [ ]:
# Verified API-Football player photos
verified_photos = {
    "Alex Iwobi": 1455,
    "Alexander Isak": 2864,
    "Antoine Semenyo": 19281,
    "Bruno Fernandes": 1485,
    "Bukayo Saka": 1460,
    "Declan Rice": 2937,
    "Erling Haaland": 1100,
}

# Add verified photos to our working DataFrame
for player_name, api_id in verified_photos.items():

    mask = photo_candidates["player_name"] == player_name

    photo_candidates.loc[mask, "api_football_id"] = api_id

    photo_candidates.loc[mask, "photo_url"] = (
        f"https://media.api-sports.io/football/players/{api_id}.png"
    )

    photo_candidates.loc[mask, "match_status"] = "matched"

print(photo_candidates["match_status"].value_counts())

print(
    photo_candidates.loc[
        photo_candidates["match_status"] == "matched",
        ["player_name", "api_football_id", "photo_url"]
    ].to_string(index=False)
)

In [ ]:
import time

# Select the next 10 unmatched players
batch = photo_candidates[
    photo_candidates["match_status"] == "pending"
].head(10)

for _, row in batch.iterrows():

    player_name = row["player_name"]
    search_name = row["search_name"]

    response = requests.get(
        "https://v3.football.api-sports.io/players",
        headers=headers_api_football,
        params={
            "search": search_name,
            "league": 39,
            "season": 2024
        },
        timeout=30
    )

    print(f"\n--- {player_name} ---")

    if response.status_code != 200:
        print("HTTP error:", response.status_code)
        break

    result = response.json()

    if result.get("errors"):
        print("API errors:", result["errors"])
        break

    matches = result.get("response", [])

    if not matches:
        print("NOT FOUND")
    else:
        for item in matches[:5]:
            p = item["player"]

            print(
                p.get("id"),
                "|", p.get("name"),
                "|", p.get("birth", {}).get("date"),
                "|", p.get("nationality")
            )

    time.sleep(2)

In [ ]:
# Newly verified player photos
new_verified_photos = {
    "Bryan Mbeumo": 20589,
    "Caoimhin Kelleher": 281,
    "Cole Palmer": 152982,
    "David Raya": 19465,
}

for player_name, api_id in new_verified_photos.items():

    mask = photo_candidates["player_name"] == player_name

    photo_candidates.loc[mask, "api_football_id"] = api_id

    photo_candidates.loc[mask, "photo_url"] = (
        f"https://media.api-sports.io/football/players/{api_id}.png"
    )

    photo_candidates.loc[mask, "match_status"] = "matched"

# Mark players who need an alternative search
needs_review = [
    "Adrien Truffert",
    "Anan Khalaili",
    "António Silva",
    "Bobby Thomas",
]

photo_candidates.loc[
    photo_candidates["player_name"].isin(needs_review),
    "match_status"
] = "review"

print(photo_candidates["match_status"].value_counts())

In [ ]:
# Select the next 10 pending players
batch = photo_candidates[
    photo_candidates["match_status"] == "pending"
].head(10)

for _, row in batch.iterrows():

    player_name = row["player_name"]

    import re
    import unicodedata

    search_name = row["search_name"]

    # Remove accents (e.g., António → Antonio)
    search_name = "".join(
        char for char in unicodedata.normalize("NFKD", search_name)
        if not unicodedata.combining(char)
    )

    # Replace hyphens and other special characters with spaces
    search_name = re.sub(r"[^a-zA-Z0-9 ]", " ", search_name)

    # Remove extra spaces
    search_name = " ".join(search_name.split())

    try:
        response = requests.get(
            "https://v3.football.api-sports.io/players",
            headers=headers_api_football,
            params={
                "search": search_name,
                "league": 39,
                "season": 2024
            },
            timeout=30
        )

        print(f"\n{player_name}:")

        if response.status_code != 200:
            print(f"  HTTP error: {response.status_code}")
            break

        result = response.json()

        if result.get("errors"):
            print("  API error:", result["errors"])
            break

        matches = result.get("response", [])

        if not matches:
            print("  NOT FOUND")
        else:
            for item in matches[:5]:
                p = item["player"]

                print(
                    f"  {p['id']} | {p['name']} | "
                    f"{p.get('birth', {}).get('date')}"
                )

    except requests.RequestException as e:
        print("Request failed:", e)
        break

    time.sleep(2)